## ATNLP Lab 4 – Basic Retrieval-Augmented Generation

If you have been following the tutorials, you should now know how to:

- Format prompts dynamically using `langchain`.

- Evaluate GPT-2 on an MCQA task.

- Do semantic search over a PDF using `sentence-transformers`.

In this lab, the objective is to compare prompt strategies and understand differences between a pre-trained model and an instruction-tuned model.

In [ ]:
%pip install -q transformers datasets sentence-transformers numpy

### 0. Environment setup

Run the installation cell above before executing the rest of the notebook. If your environment already has these packages, you can skip it.

GPT-2 is a pre-trained language model, so it often performs better when prompts resemble natural text.
In the next cell, we build an "internet-style" prompt from an MMLU sample to test whether prompt style alone changes performance.

For consistency with Notebook 1, we keep the same MCQA structure (`A.`, `B.`, `C.`, `D.` and `Answer:`) while changing only the surrounding wording.

In [ ]:
def _normalize_mmlu_choices(choices):
    if isinstance(choices, dict):
        choices = list(choices.values())
    return list(choices)

def convert_to_internet_prompt(sample: dict) -> str:
    """Convert an MMLU sample into a prompt that more naturally resembles internet text."""
    question = sample["question"].strip()
    choices = _normalize_mmlu_choices(sample["choices"])
    option_labels = ["A", "B", "C", "D"]

    options_block = "\n".join(
        [f"{label}. {choice}" for label, choice in zip(option_labels, choices)]
    )

    prompt = f"""I found this study question while browsing a forum and wanted a quick answer.

{question}

{options_block}

Answer:"""
    return prompt

Now, instead of changing only prompt style, we switch to an instruction-tuned model: [vicgalle/gpt2-open-instruct-v1](https://huggingface.co/vicgalle/gpt2-open-instruct-v1).

This model expects an instruction/response format, so we map MMLU items into that template and compare behavior against the same task formulation.

Model card prompt format:
```
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Pretend you are an alien visiting Earth. Write three opinions you believe.

### Response:
```
In the next code cell, we implement that converter for MMLU.

In [ ]:
def convert_to_instruction_prompt(sample: dict) -> str:
    """Convert an MMLU sample into an instruction prompt that fits vicgalle/gpt2-open-instruct-v1."""
    question = sample["question"].strip()
    choices = _normalize_mmlu_choices(sample["choices"])
    option_labels = ["A", "B", "C", "D"]

    options_block = "\n".join(
        [f"{label}. {choice}" for label, choice in zip(option_labels, choices)]
    )

    instruction = f"""Read the multiple-choice question and select the single best option.

{question}

{options_block}

Return only the option letter: A, B, C, or D."""

    prompt = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
"""
    return prompt

Next, we add retrieval-augmented generation (RAG): for each MMLU question, retrieve relevant chunks from Lab 2 artifacts and append them to the instruction prompt.

This gives the model extra factual context at inference time, without retraining.

Pipeline for this section:
1. Load `chunks.pkl` and `embeddings.pkl` from Lab 2.
2. Retrieve top-k chunks using cosine similarity with a query embedding.
3. Inject retrieved context into the instruction prompt.
4. Evaluate instruction-only vs instruction+RAG on the same model.

In [ ]:
def convert_to_prompt_with_context(sample: dict) -> str:
    """Convert an MMLU sample into an instruction prompt augmented with retrieved context."""
    question = sample["question"].strip()
    choices = _normalize_mmlu_choices(sample["choices"])
    option_labels = ["A", "B", "C", "D"]

    options_block = "\n".join(
        [f"{label}. {choice}" for label, choice in zip(option_labels, choices)]
    )

    retrieved_context = sample.get("retrieved_context", [])
    if isinstance(retrieved_context, str):
        retrieved_context = [retrieved_context]

    context_block = "\n\n".join(
        [f"Context {idx + 1}: {chunk}" for idx, chunk in enumerate(retrieved_context)]
    ) or "No external context available."

    instruction = f"""Use the retrieved context when helpful to answer the question.

{question}

{options_block}

Retrieved context:
{context_block}

Return only the option letter: A, B, C, or D."""

    prompt = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
"""
    return prompt

### 8. Running GPT-2-instruct on MMLU

We now define helper utilities for generation and evaluation.
The key idea is to keep decoding deterministic (`do_sample=False`) and extract only the answer letter (`A/B/C/D`) from model outputs so comparisons are consistent.

In [ ]:
from pathlib import Path
import pickle
import re
from typing import List, Tuple

import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

In [ ]:
option_labels = ["A", "B", "C", "D"]

def load_generation_model(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name)
    return tokenizer, model

def generate_response(prompt: str, tokenizer, model, max_new_tokens: int = 16) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated[len(prompt):].strip()

def extract_answer_letter(text: str) -> str:
    matches = re.findall(r"\b([ABCD])\b", text.upper())
    return matches[-1] if matches else ""

### 9. Reusing Lab 2 artifacts (chunks + embeddings) for RAG

From Lab 2, we reuse `data/biology/chunks.pkl` and `data/biology/embeddings.pkl` as a lightweight knowledge base.
For each MMLU question, we encode the question, retrieve top-k similar chunks with cosine similarity, and attach those chunks to the prompt as external context.
To stay consistent with Notebook 2, this section keeps a `search(...)` helper (same spirit/API) and uses it through a RAG wrapper.

In [ ]:
def unpickle_file(file_path: Path):
    with open(file_path, "rb") as f:
        return pickle.load(f)

def load_lab2_artifacts(base_paths: List[Path]) -> Tuple[List[str], np.ndarray]:
    for base in base_paths:
        chunks_path = base / "chunks.pkl"
        embeddings_path = base / "embeddings.pkl"
        if chunks_path.exists() and embeddings_path.exists():
            chunks = unpickle_file(chunks_path)
            embeddings = np.array(unpickle_file(embeddings_path))
            print(f"Loaded artifacts from: {base}")
            print(f"Chunks: {len(chunks)} | Embeddings shape: {embeddings.shape}")
            return chunks, embeddings
    raise FileNotFoundError(
        "Could not find Lab 2 artifacts. Run Lab 2 and make sure chunks.pkl and embeddings.pkl exist."
    )

possible_paths = [
    Path("data/biology"),
    Path("../data/biology"),
    Path("../../data/biology"),
]

chunks, embeddings = load_lab2_artifacts(possible_paths)

retriever_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

def retrieve_top_k_context(
    query: str,
    chunks: List[str],
    embeddings: np.ndarray,
    retriever_model: SentenceTransformer,
    k: int = 3,
    ) -> List[Tuple[str, float]]:
    query_embedding = retriever_model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    embeddings = np.asarray(embeddings)
    if embeddings.ndim != 2:
        raise ValueError("Embeddings must be a 2D matrix of shape (n_chunks, dim).")

    if np.linalg.norm(embeddings[0]) < 0.99 or np.linalg.norm(embeddings[0]) > 1.01:
        emb_norm = np.linalg.norm(embeddings, axis=1, keepdims=True)
        embeddings = embeddings / np.clip(emb_norm, 1e-12, None)

    similarities = embeddings @ query_embedding
    top_indices = np.argsort(similarities)[-k:][::-1]
    return [(chunks[i], float(similarities[i])) for i in top_indices]

def search(
    query: str,
    embeddings: np.ndarray,
    chunks: List[str],
    model: SentenceTransformer,
    k: int = 5,
    ) -> List[Tuple[str, float]]:
    """Notebook 2-compatible retrieval helper."""
    return retrieve_top_k_context(
        query=query,
        chunks=chunks,
        embeddings=embeddings,
        retriever_model=model,
        k=k,
    )

In [ ]:
def build_rag_sample(
    sample: dict,
    chunks: List[str],
    embeddings: np.ndarray,
    retriever_model: SentenceTransformer,
    k: int = 3,
    ) -> dict:
    retrieved = retrieve_top_k_context(
        sample["question"],
        chunks=chunks,
        embeddings=embeddings,
        retriever_model=retriever_model,
        k=k,
    )
    sample_with_context = dict(sample)
    sample_with_context["retrieved_context"] = [text for text, _ in retrieved]
    sample_with_context["retrieval_scores"] = [score for _, score in retrieved]
    return sample_with_context

# Quick retrieval sanity check
example_query = "What is photosynthesis?"
retrieved_example = retrieve_top_k_context(
    example_query,
    chunks=chunks,
    embeddings=embeddings,
    retriever_model=retriever_model,
    k=3,
    )

for i, (chunk, score) in enumerate(retrieved_example, start=1):
    print(f"[{i}] score={score:.4f}")
    print(chunk[:250], "...\n")

### 10. Mini comparison on MMLU: instruction vs instruction+RAG

This final experiment uses the **same model** (`vicgalle/gpt2-open-instruct-v1`) for both settings:

- Instruction-only prompt
- Instruction prompt + retrieved context (RAG)

This isolates the effect of retrieval, since the model and decoding setup are unchanged.

In [ ]:
def evaluate_prompt_function(
    dataset_split,
    prompt_fn,
    tokenizer,
    model,
    n_samples: int = 30,
    ) -> float:
    correct = 0
    total = min(n_samples, len(dataset_split))

    for idx in range(total):
        sample = dataset_split[idx]
        prompt = prompt_fn(sample)
        response = generate_response(prompt, tokenizer, model)
        pred_letter = extract_answer_letter(response)
        gold_letter = option_labels[sample["answer"]]
        correct += int(pred_letter == gold_letter)

    return correct / total if total else 0.0

# Load a small MMLU slice (choose another subject if you want)
mmlu = load_dataset("cais/mmlu", "high_school_biology")
test_split = mmlu["test"]

# Same model in both conditions
instruct_model_name = "vicgalle/gpt2-open-instruct-v1"
instr_tokenizer, instr_model = load_generation_model(instruct_model_name)

# Build RAG-aware wrapper for the prompt function
def rag_prompt_fn(sample: dict) -> str:
    sample_with_context = build_rag_sample(
        sample,
        chunks=chunks,
        embeddings=embeddings,
        retriever_model=retriever_model,
        k=3,
    )
    return convert_to_prompt_with_context(sample_with_context)

n_eval = 30
acc_instruction_only = evaluate_prompt_function(
    test_split, convert_to_instruction_prompt, instr_tokenizer, instr_model, n_samples=n_eval
    )
acc_instruction_rag = evaluate_prompt_function(
    test_split, rag_prompt_fn, instr_tokenizer, instr_model, n_samples=n_eval
    )

print(f"GPT2-instruct + instruction prompt accuracy (@{n_eval}): {acc_instruction_only:.3f}")
print(f"GPT2-instruct + instruction+RAG prompt accuracy (@{n_eval}): {acc_instruction_rag:.3f}")